# Lab 1: Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 4: Evaluation

This notebook scores the fine-tuned model on the 123 held-out contracts and compares three
models on the same task:

| model | accuracy | evidence F1 | contradiction F1 | valid JSON |
| ----- | -------- | ----------- | ---------------- | ---------- |
| Nemotron 3 Nano 4B, untuned | 13.6 | 7.8 | 3.8 | 30.1 |
| **the same 4B, fine-tuned** | **87.9** | **76.6** | **70.1** | **100.0** |
| Claude Sonnet 5 | 84.1 | 68.6 | 62.3 | 100.0 |

A 4B model fine-tuned on 423 contracts scores above Sonnet 5 on this task, and it runs on one
A10G.

All three rows ship with the lab as pre-computed results. Part 1 can re-score the middle row
against your own endpoint; it will differ by a few tenths, because requests are greedy but vLLM
batches whatever is in flight.

Two methods:

| Method | What it is good for |
| ------ | ------------------- |
| **Statistical scoring** | exact, reproducible, free. Was the verdict right? Were the cited spans right? |
| **LLM-as-a-Judge** | semantic. Does the cited clause actually support the verdict, even when it is not the span the annotator picked? |

Parts 1 and 2 are the statistical scoring, done by `contractnli_scorer` over all 123 contracts.
Part 3 is the judge, run as a managed Amazon Bedrock evaluation job over a 30-contract slice.

### Setup

In [ ]:
import hashlib
import json
import os
import statistics
import time
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

import boto3
from botocore.config import Config
from sagemaker.core.helper.session_helper import Session, get_execution_role

import contractnli as C
from contractnli_scorer import score_record
from config import BASE_MODEL_ID, MODEL_SLUG

# The endpoint notebook 3 created. Its name is derived exactly as notebook 3 derives it, so
# the two agree without copying a string between notebooks.
MAX_NAME = 63


def rname(base, suffix):
    cand = f"{base}{suffix}"
    if len(cand) <= MAX_NAME:
        return cand
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    keep = MAX_NAME - len(suffix) - len(digest) - 1
    return f"{base[:keep].rstrip('-')}-{digest}{suffix}"


ENDPOINT_NAME = rname(f"{MODEL_SLUG}-contractnli", "-sft-ep")
SERVED_MODEL_NAME = "nemotron-contractnli"   # SM_VLLM_SERVED_MODEL_NAME in notebook 3

# how many contracts to score, and how many requests to keep in flight. vLLM batches
# concurrent requests, so this is the difference between ~15 minutes and ~3.
RECORDS = 123
WORKERS = 8

BASELINES = "baselines"          # pre-computed results that ship with the lab

# Two switches, because the two evaluations cost very different amounts of time.
# Either can be left False to read that part's results from BASELINES instead.
RUN_ENDPOINT_EVAL = False   # score your endpoint over 123 contracts: about 3 minutes
RUN_JUDGE_EVAL = False      # two Bedrock judge jobs over 30 contracts: about 20 minutes

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(
        RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket_name = sess.default_bucket()
region = sess.boto_region_name

runtime = boto3.client("sagemaker-runtime", region_name=region,
                       config=Config(read_timeout=900, retries={"max_attempts": 0}))
print(f"endpoint: {ENDPOINT_NAME}  region: {region}")
print(f"bucket:   {bucket_name}")

In [ ]:
C.ensure_dataset("./data")
EVAL_DOCS, labels = C.load("test")
EVAL_DOCS = EVAL_DOCS[:RECORDS]
label_keys = list(labels.keys())


def gold_json(doc):
    """The expert answer, in the shape the model is trained to emit."""
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})


print(f"{len(EVAL_DOCS)} contracts, {len(label_keys)} checklist items each")

### What we measure

`contractnli_scorer.score_record` compares the model's JSON against the annotator's, per
contract:

- **accuracy**: the share of the 17 verdicts that match.
- **evidence F1**: how well the cited span numbers match the annotator's, so a model cannot
  score well by guessing labels without pointing at the text.
- **contradiction F1**: the same, restricted to `Contradiction`. This is the rarest and
  hardest class, and the one where models differ most.
- **valid JSON**: whether the answer parsed at all. Read this first: the other three are
  meaningless if it is low.

In [ ]:
# The scorer only accepts a reference that starts with '{'. Anything else silently scores
# 0.00 on every content metric while valid JSON still reads 100, because that flag reflects
# only the model's own output. Assert the contract rather than trusting it.
_probe = gold_json(EVAL_DOCS[0])
assert _probe.lstrip().startswith("{"), f"gold is not bare JSON: {_probe[:60]!r}"

_perfect = score_record({"id": "check", "model_response": _probe,
                         "reference_answer": _probe}, 0)
_flat = {m["name"]: m["value"] for m in _perfect["metrics_list"]}
assert _flat["label_correct"] == 1.0, f"scoring gold against itself gave {_flat}"
print("scorer self-check passed: gold scored against itself is 1.0")

### Use pre-computed results, or run the evaluations yourself

Both switches default to `False`, which reads pre-computed results from `baselines/` and makes no
calls. Set a switch to `True` to run that evaluation yourself.

| Switch | `False` (default) | `True` | Time when `True` |
| ------ | ----------------- | ------ | ---------------- |
| `RUN_ENDPOINT_EVAL` | reads pre-computed results | scores all 123 contracts against your endpoint | ~3 min |
| `RUN_JUDGE_EVAL` | reads pre-computed results | runs two Bedrock judge jobs over 30 contracts | ~20 min |

The switches are independent. `RUN_JUDGE_EVAL = True` with `RUN_ENDPOINT_EVAL = False` judges the
pre-computed answers, so it needs no endpoint.

Most of the judge's 20 minutes is fixed startup: a job over two contracts takes eight.

## Part 1: Score the fine-tuned model against your endpoint

Each request sends one contract and asks for the 17 verdicts. `chat_template_kwargs` carries
the same reasoning-off switch that training used, so the model answers directly instead of
deliberating; `max_tokens` of 700 is roughly triple what a complete answer needs.

In [ ]:
def ask_endpoint(doc, max_tokens=700):
    """One contract to the endpoint, returning the raw generated text."""
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": C.build_messages(doc, labels),
        "max_tokens": max_tokens,
        "temperature": 0,
        # the flag that turns reasoning off, forwarded per request exactly as in training
        "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS,
    }
    r = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                                ContentType="application/json",
                                Body=json.dumps(payload))
    return json.loads(r["Body"].read())["choices"][0]["message"]["content"]


# one contract first, so a misconfigured endpoint fails in seconds rather than minutes
if RUN_ENDPOINT_EVAL:
    _t = time.time()
    _sample = ask_endpoint(EVAL_DOCS[0])
    print(f"{time.time() - _t:.1f}s, {len(_sample)} chars\n")
    print(_sample[:400], "...")
else:
    print("RUN_ENDPOINT_EVAL is False, so the endpoint is not called")

In [ ]:
def score_all(docs, workers=WORKERS):
    """Score every contract, keeping `workers` requests in flight."""
    def one(i_doc):
        i, d = i_doc
        try:
            return i, ask_endpoint(d)
        except Exception as e:                              # noqa: BLE001
            print(f"  record {i}: {type(e).__name__}")
            return i, ""

    t0, gens = time.time(), {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        # tqdm.auto renders a widget in Jupyter and a plain bar elsewhere. ex.map yields in
        # order as results arrive, so the bar tracks completed contracts, not submitted ones.
        with tqdm(total=len(docs), unit="contract", desc=f"scoring ({workers} workers)") as bar:
            for i, text in ex.map(one, enumerate(docs)):
                gens[i] = text
                bar.update(1)
                bar.set_postfix_str(f"{(time.time() - t0) / len(gens):.1f}s/record")

    rows = []
    for i, d in enumerate(docs):
        scored = score_record({"id": str(i), "model_response": gens[i],
                               "reference_answer": gold_json(d)}, i)
        flat = {m["name"]: m["value"] for m in scored["metrics_list"]}
        rows.append({"index": i, "generated": gens[i], **flat})
    print(f"\n{len(rows)} contracts in {(time.time() - t0) / 60:.1f} min")
    return rows


def summarise(rows):
    names = ("label_correct", "evidence_f1", "contradiction_correct", "json_valid")
    return {n: round(100 * statistics.mean(r[n] for r in rows), 2) for n in names}


if RUN_ENDPOINT_EVAL:
    tuned_rows = score_all(EVAL_DOCS)
else:
    # the same generations the shipped reference metrics and judge results describe
    with open(f"{BASELINES}/tuned_reference_rows.json") as f:
        tuned_rows = json.load(f)
    print(f"loaded {len(tuned_rows)} pre-computed answers from {BASELINES}")

tuned = summarise(tuned_rows)
for k, v in tuned.items():
    print(f"  {k:24s} {v:6.2f}")

## Part 2: The other two models

The untuned model and Sonnet 5 were scored with the same scorer on the same contracts, and
their results ship with the lab.

**The untuned 4B** takes about half an hour, and most of its answers do not parse. Its failure
has one shape: it emits well-formed JSON that never ends, listing dozens of span numbers as
evidence for every checklist item until it runs out of budget.

**Sonnet 5** is a live API call per contract, and it sees a slightly different prompt: the 4B is
prompted through its own chat template with reasoning disabled, while Sonnet gets the same
system and user turns as plain messages. Sonnet also does not accept a temperature setting, so
its answers are not greedy.

In [ ]:
def load_precomputed(name):
    with open(f"{BASELINES}/{name}_metrics.json") as f:
        return json.load(f)


base = load_precomputed("base")
frontier = load_precomputed("frontier")

rows_out = ((f"{MODEL_SLUG} (untuned)", base),
            (f"{MODEL_SLUG} (fine-tuned)", tuned),
            (frontier["model"].split(".")[-1], frontier))
w = max(len(n) for n, _ in rows_out) + 2
print(f"{'model':{w}s} {'accuracy':>9s} {'evidence-F1':>12s} {'contra-F1':>10s} {'JSON':>7s}")
for name, m in rows_out:
    print(f"{name:{w}s} {m['label_correct']:9.1f} {m['evidence_f1']:12.1f} "
          f"{m['contradiction_correct']:10.1f} {m['json_valid']:7.1f}")

# A reference run of this exact lab, for comparison. Yours will differ by a few tenths.
ref = load_precomputed("tuned_reference")
print()
print(f"{'reference run of this lab':{w}s} {ref['label_correct']:9.1f} "
      f"{ref['evidence_f1']:12.1f} {ref['contradiction_correct']:10.1f} {ref['json_valid']:7.1f}")
if RUN_ENDPOINT_EVAL:
    print(f"{'your run, difference':{w}s} {tuned['label_correct'] - ref['label_correct']:+9.1f} "
          f"{tuned['evidence_f1'] - ref['evidence_f1']:+12.1f} "
          f"{tuned['contradiction_correct'] - ref['contradiction_correct']:+10.1f} "
          f"{tuned['json_valid'] - ref['json_valid']:+7.1f}")

### The same numbers as a chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BASE_C, FRONTIER_C, TUNED_C = "#b0b0b0", "#ff7f0e", "steelblue"
metrics = ["accuracy", "evidence F1", "contradiction F1"]
series = [
    ("untuned", BASE_C, [base["label_correct"], base["evidence_f1"],
                         base["contradiction_correct"]]),
    ("fine-tuned", TUNED_C, [tuned["label_correct"], tuned["evidence_f1"],
                             tuned["contradiction_correct"]]),
    ("Sonnet 5", FRONTIER_C, [frontier["label_correct"], frontier["evidence_f1"],
                              frontier["contradiction_correct"]]),
]

x = np.arange(len(metrics))
width = 0.26
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, (label, colour, vals) in enumerate(series):
    bars = ax.bar(x + (k - 1) * width, vals, width, label=label, color=colour)
    ax.bar_label(bars, fmt="%.1f", fontsize=9, padding=2)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("%"); ax.set_ylim(0, 100)
ax.legend(); ax.set_title("ContractNLI, 123 held-out contracts")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Part 3: LLM-as-a-Judge

The scoring above compares span **numbers**. That is exact and free, and unfair in one specific
way: if the model cites clause 47 and the annotator cited 48, and both say the same thing, the
scorer records a miss. With `evidence_f1` in the seventies, part of what is left is citations a
lawyer would accept and a string comparison will not.

A judge model reads the cited clause text and decides whether it justifies the verdict. These are
the same three custom metrics the serverless version of this workshop uses, unchanged, so the
numbers are comparable between the two labs.

The route differs. That lab hands a registered Model Package to `LLMAsJudgeEvaluator`, which runs
inference for you. Notebook 2 registers no Model Package, so here the endpoint has already
produced the answers and they go to Bedrock as **precomputed inference responses**. Bedrock
judges; it never loads the model.

> **IAM requirements.** The judge runs on Amazon Bedrock, so the execution role needs both:
>
> 1. a **trust relationship** allowing `bedrock.amazonaws.com` to assume it, and
> 2. an **identity-based policy** granting `bedrock:CreateEvaluationJob`, `GetEvaluationJob`,
>    `ListEvaluationJobs`, `StopEvaluationJob`, and `bedrock:InvokeModel` on the evaluator
>    model's inference profile.
>
> See [Bedrock evaluation permissions](https://docs.aws.amazon.com/bedrock/latest/userguide/judge-service-roles.html).

In [ ]:
# Sonnet 4.5 has no on-demand throughput, so an evaluation job needs the inference profile that
# fronts it rather than the bare model id. The bare id raises ValidationException: Invocation of
# model ID ... with on-demand throughput isn't supported.
EVALUATOR_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# The judge is billed per token and reads a whole contract per record, so it runs on a slice.
# The statistical scoring above still uses all 123.
JUDGE_N = 30

JUDGE_PREFIX = "contractnli-judge-eval"
JUDGE_OUTPUT = f"s3://{bucket_name}/{JUDGE_PREFIX}/output"

_intro = (
    "Judge ONLY the final JSON object in the response. These models emit a "
    "<think> reasoning block first; ignore it completely, a confident-sounding "
    "deliberation that reaches the wrong label is still wrong. The response may "
    "also arrive wrapped in a stringified list; unwrap it.\n"
)

custom_metrics = [
    {"customMetricDefinition": {
        "name": "EvidenceSupportsVerdict",
        "instructions": (
            _intro +
            "The contract is given as numbered spans in the prompt. For each checklist "
            "item the response gives a verdict and cites span numbers. Look up the cited "
            "spans in the prompt and judge whether their text actually justifies the "
            "verdict, ignore whether they match the reference span numbers, since a "
            "different clause may state the same thing.\n"
            "Prompt: {{prompt}}\nResponse: {{prediction}}\nReference: {{ground_truth}}"),
        "ratingScale": [
            {"definition": "Every committed verdict is supported by the clauses it cites",
             "value": {"floatValue": 3}},
            {"definition": "Most verdicts are supported; a few citations are off",
             "value": {"floatValue": 2}},
            {"definition": "Several citations do not support their verdict",
             "value": {"floatValue": 1}},
            {"definition": "Citations are largely unrelated to the verdicts",
             "value": {"floatValue": 0}},
        ]}},
    {"customMetricDefinition": {
        "name": "CarveOutHandling",
        "instructions": (
            _intro +
            "A checklist item stated absolutely is contradicted by a clause with no "
            "exception for it. Judge whether the response handled exceptions and "
            "carve-outs correctly, rather than defaulting to NotMentioned when the "
            "contract is in fact silent, or to Entailment when a protection is absent.\n"
            "Prompt: {{prompt}}\nResponse: {{prediction}}\nReference: {{ground_truth}}"),
        "ratingScale": [
            {"definition": "Every exception and carve-out is handled correctly",
             "value": {"floatValue": 3}},
            {"definition": "Most are handled; one or two are missed",
             "value": {"floatValue": 2}},
            {"definition": "Several are missed, defaulting to NotMentioned or Entailment",
             "value": {"floatValue": 1}},
            {"definition": "Exceptions are ignored throughout", "value": {"floatValue": 0}},
        ]}},
    {"customMetricDefinition": {
        "name": "ChecklistCompleteness",
        "instructions": (
            _intro +
            "Check that the response is valid JSON containing an entry for every "
            "checklist key listed in the prompt, each with a label and an evidence "
            "list, and no commentary outside the JSON.\n"
            "Prompt: {{prompt}}\nResponse: {{prediction}}"),
        "ratingScale": [
            {"definition": "Complete and well-formed", "value": {"floatValue": 1}},
            {"definition": "Missing entries or malformed", "value": {"floatValue": 0}},
        ]}},
]
METRIC_NAMES = [m["customMetricDefinition"]["name"] for m in custom_metrics]
print(f"{len(custom_metrics)} custom metrics, judged by {EVALUATOR_MODEL}")

### Hand the answers to Bedrock

One JSONL record per contract, in the shape a precomputed-inference job expects: the prompt the
model saw, the answer it gave, and the annotator's reference. The `{{prompt}}`, `{{prediction}}`
and `{{ground_truth}}` placeholders in the instructions above resolve to those three fields.

In [ ]:
def build_judge_dataset(rows, name, n=JUDGE_N):
    """Upload `n` answers as precomputed inference responses, returning the S3 URI."""
    records = []
    for r in rows[:n]:
        doc = EVAL_DOCS[r["index"]]
        records.append({
            # the judge has to look up cited span numbers, so it gets the user turn verbatim
            "prompt": C.build_messages(doc, labels)[1]["content"],
            "referenceResponse": gold_json(doc),
            "modelResponses": [{"response": r["generated"], "modelIdentifier": name}],
        })
    key = f"{JUDGE_PREFIX}/input/{name}-{n}.jsonl"
    body = "".join(json.dumps(rec) + "\n" for rec in records)
    boto3.client("s3", region_name=region).put_object(
        Bucket=bucket_name, Key=key, Body=body.encode())
    uri = f"s3://{bucket_name}/{key}"
    print(f"{name}: {len(records)} records -> {uri}")
    return uri


def launch_judge(uri, name, n=JUDGE_N):
    """Create the Bedrock evaluation job and return its ARN."""
    bedrock = boto3.client("bedrock", region_name=region)
    job = bedrock.create_evaluation_job(
        jobName=f"contractnli-judge-{name}-{n}-{int(time.time())}",
        roleArn=role,
        applicationType="ModelEvaluation",
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "General",
                "dataset": {"name": "ContractNLI", "datasetLocation": {"s3Uri": uri}},
                "metricNames": METRIC_NAMES,
            }],
            "customMetricConfig": {
                "customMetrics": custom_metrics,
                "evaluatorModelConfig": {
                    "bedrockEvaluatorModels": [{"modelIdentifier": EVALUATOR_MODEL}]},
            },
        }},
        # precomputedInferenceSource is what lets a model Bedrock cannot host be judged here
        inferenceConfig={"models": [
            {"precomputedInferenceSource": {"inferenceSourceIdentifier": name}}]},
        outputDataConfig={"s3Uri": JUDGE_OUTPUT},
    )
    print(f"launched {name}: {job['jobArn']}")
    return job["jobArn"]


def wait_for_judge(arns, poll=60):
    """Block until every job leaves InProgress, reporting elapsed minutes as it goes."""
    bedrock = boto3.client("bedrock", region_name=region)
    pending, started = dict(arns), time.time()
    while pending:
        for name, arn in list(pending.items()):
            job = bedrock.get_evaluation_job(jobIdentifier=arn)
            if job["status"] not in ("InProgress", "Scheduled", "Stopping"):
                print(f"[{(time.time() - started) / 60:5.1f} min] {name}: {job['status']}")
                if job["status"] == "Failed":
                    print(f"  {job.get('failureMessages')}")
                pending.pop(name)
        if pending:
            print(f"[{(time.time() - started) / 60:5.1f} min] waiting on {', '.join(pending)}")
            time.sleep(poll)


def read_judge_scores(arn):
    """Per-metric scores from a finished job, keeping None where the judge declined.

    None is not zero. The judge returns nothing when it cannot read an answer, and counting that
    as zero charges a model twice: once for the metric it failed, once for the zero. Those are
    kept out of the mean and surfaced as coverage instead.
    """
    bedrock = boto3.client("bedrock", region_name=region)
    s3 = boto3.client("s3", region_name=region)
    out = bedrock.get_evaluation_job(jobIdentifier=arn)["outputDataConfig"]["s3Uri"]
    prefix = out.replace(f"s3://{bucket_name}/", "")
    job_id = arn.split("/")[-1]
    scores = {}
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(".jsonl") or job_id not in obj["Key"]:
                continue
            body = s3.get_object(Bucket=bucket_name, Key=obj["Key"])["Body"].read().decode()
            for line in body.splitlines():
                if not line.strip():
                    continue
                result = json.loads(line).get("automatedEvaluationResult", {})
                for score in result.get("scores", []):
                    scores.setdefault(score["metricName"], []).append(score.get("result"))
    return scores

### Run it

`RUN_JUDGE_EVAL`, set in Setup, governs this part. Left `False` the notebook reads judge results
that ship with the lab. Set it `True` to run the judge yourself: two jobs, about 20 minutes, billed
per token against the evaluator model.

Both models are judged. The judge reads answers that already exist, either the ones Part 1 produced
or the ones that ship with the lab, so no endpoint is needed.

In [ ]:
JUDGE_FILE = f"{BASELINES}/judge_reference_metrics.json"

if RUN_JUDGE_EVAL:
    with open(f"{BASELINES}/base_rows.json") as f:
        base_rows = json.load(f)
    arns = {
        "base": launch_judge(build_judge_dataset(base_rows, "base"), "base"),
        "tuned": launch_judge(build_judge_dataset(tuned_rows, "tuned"), "tuned"),
    }
    wait_for_judge(arns)
    judge_scores = {name: read_judge_scores(arn) for name, arn in arns.items()}
else:
    with open(JUDGE_FILE) as f:
        judge_scores = json.load(f)["scores"]
    print(f"loaded pre-computed judge results from {JUDGE_FILE}")

In [ ]:
def judge_table(scores_by_model):
    """Mean and coverage per metric per model."""
    metrics = sorted({m for s in scores_by_model.values() for m in s})
    width = max(len(m) for m in metrics) + 2
    print(f"{'metric':{width}s}" + "".join(f"{n:>22s}" for n in scores_by_model))
    for metric in metrics:
        row = f"{metric:{width}s}"
        for name in scores_by_model:
            vals = scores_by_model[name].get(metric, [])
            got = [v for v in vals if v is not None]
            mean = sum(got) / len(got) if got else float("nan")
            row += f"{mean:.3f} ({len(got)}/{len(vals)})".rjust(22)
        print(row)


judge_table(judge_scores)

### Reading the judge

Two things about these numbers.

**They are normalised to 0-1.** A metric declared on a 0-3 scale comes back as 0, 0.333, 0.667 or
1.0, so `EvidenceSupportsVerdict` at 0.41 means 1.2 out of 3, not 41% of anything.

**Coverage is printed beside every mean.** Where it is low, or unequal between the two models, the
means describe different subsets of contracts and the gap between them is not a before and after
over the same work.

`ChecklistCompleteness` is where the judge and the statistical scorer meet most directly: it asks
whether the answer is well-formed JSON covering all 17 items, which is what `json_valid` measures
by parsing. The base model fails both.

In [ ]:
import numpy as np

metrics = sorted({m for s in judge_scores.values() for m in s})
JUDGE_COLOURS = {"base": BASE_C, "tuned": TUNED_C}

x = np.arange(len(metrics))
width = 0.36
fig, ax = plt.subplots(figsize=(9, 4.6))
for k, (name, scores) in enumerate(judge_scores.items()):
    means, coverage = [], []
    for metric in metrics:
        vals = scores.get(metric, [])
        got = [v for v in vals if v is not None]
        means.append(sum(got) / len(got) if got else np.nan)
        coverage.append(f"{len(got)}/{len(vals)}")
    bars = ax.bar(x + (k - 0.5) * width, means, width, label=name,
                  color=JUDGE_COLOURS.get(name, "grey"))
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in means], fontsize=9, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=8)
ax.set_ylabel("mean score (0-1)")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title(f"LLM-as-a-Judge over {JUDGE_N} contracts")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Putting the two together

| | Statistical scorer | LLM-as-a-Judge |
| --- | --- | --- |
| Reproducible | identical every run | varies between runs |
| Cost | free | pays per token, on long prompts |
| Judges semantic citation validity | no, span numbers only | yes |
| Can be gamed | no | prompt injection is a real risk |
| Good for | regression gates, tracking a model over time | understanding why a model is wrong |

Use the statistical metrics as the gate and the judge to explain the residual. If the judge scores
`EvidenceSupportsVerdict` high while `evidence_f1` is low, the model is citing valid clauses the
annotator did not pick, which is a labelling artefact rather than a model failure.

### Where the errors are

In [ ]:
from collections import Counter


def error_directions(rows, docs, top=6):
    """Which gold -> predicted confusions dominate."""
    pairs = Counter()
    for r, d in zip(rows, docs):
        try:
            pred = json.loads(r["generated"])
        except Exception:                                   # noqa: BLE001
            continue
        g = C.gold_for(d)
        for k, v in g.items():
            if k in pred and isinstance(pred[k], dict):
                got = pred[k].get("label")
                if got and got != v["choice"]:
                    pairs[f"{v['choice']} -> {got}"] += 1
    return pairs.most_common(top)


print("the fine-tuned model's most common confusions:")
for pair, n in error_directions(tuned_rows, EVAL_DOCS):
    print(f"  {pair:34s} {n}")

### Takeaways

**Fine-tuning a 4B beat a frontier model on this task.** Not in general, on this task: a fixed
17-item checklist over a document type the model saw 423 examples of, where the answer has to
cite spans.

**Valid JSON is the first thing to read.** The untuned model's low scores are mostly not wrong
verdicts, they are answers that never finished. Without valid JSON the other three metrics
cannot be measured.

**Contradiction is the hardest class** for every model here, and the gap between the tuned model
and Sonnet is widest there. If this were going to production, that is the class to look at.

**The two methods agree.** `ChecklistCompleteness` from the judge and `json_valid` from the scorer
measure the same property by different means, and both put the untuned model near 0.30 and the
fine-tuned one at 1.00.

## Clean up

> **Important:** a real-time endpoint bills per instance-hour for as long as it exists,
> whether or not you send it traffic.

Three resources, in dependency order: endpoint, then config, then model. The names are
rebuilt with the same helper notebook 3 used, since a fresh kernel shares no state.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

stem = f"{MODEL_SLUG}-contractnli"
for label, fn in [
    ("endpoint", lambda: Endpoint.get(ENDPOINT_NAME).delete()),
    ("endpoint config", lambda: EndpointConfig.get(rname(stem, "-sft-cfg")).delete()),
    ("model", lambda: Model.get(rname(stem, "-sft-m")).delete()),
]:
    try:
        fn()
        print(f"deleted {label}")
    except Exception as error:
        print(f"skip {label}: {error}")